# Instagram non-overlapping 3-year periods

This notebook builds the static 3-year RCEDTG pipeline with one row per user and period.

## Setup

We define paths, constants, tokenization helpers, and the three non-overlapping periods.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd


OUTPUT_DIR = Path.cwd()
DATA_DIR = OUTPUT_DIR.parents[1]

COMMENTS_SOURCE = DATA_DIR / "ig_comments_clean.csv"
POSTS_SOURCE = DATA_DIR / "ig_posts_clean.csv"

COMMENTS_OUTPUT = OUTPUT_DIR / "ig_comments_top_start.csv"
POSTS_OUTPUT = OUTPUT_DIR / "ig_posts_top_start.csv"
METRICS_OUTPUT = OUTPUT_DIR / "ig_user_nonoverlapping_3y_periods_new_definitions.csv"

ANALYSIS_START = pd.Timestamp("2017-01-01")
ANALYSIS_END_EXCLUSIVE = pd.Timestamp("2026-04-01")
CREATOR_USERNAME = "camihawke"
MIDNIGHT_BACKOFF = pd.Timedelta(minutes=2)

COMMENT_DTYPES = {
    "comment_id": "string",
    "media_id": "string",
    "text": "string",
    "timestamp": "string",
    "from_id": "string",
    "from_username": "string",
    "parent_id": "string",
    "is_reply": "boolean",
    "is_creator": "boolean",
}

POST_DTYPES = {
    "media_id": "string",
    "timestamp": "string",
}

NON_EMOJI_TOKEN_RE = re.compile(
    r"https?://\\S+|[#@]?\\w+(?:[.'_-]\\w+)*",
    flags=re.UNICODE,
)

EMOJI_RANGES = (
    (0x00A9, 0x00A9),
    (0x00AE, 0x00AE),
    (0x203C, 0x203C),
    (0x2049, 0x2049),
    (0x2122, 0x2122),
    (0x2139, 0x2139),
    (0x2194, 0x2199),
    (0x21A9, 0x21AA),
    (0x231A, 0x231B),
    (0x2328, 0x2328),
    (0x23CF, 0x23CF),
    (0x23E9, 0x23F3),
    (0x23F8, 0x23FA),
    (0x24C2, 0x24C2),
    (0x25AA, 0x25AB),
    (0x25B6, 0x25B6),
    (0x25C0, 0x25C0),
    (0x25FB, 0x25FE),
    (0x2600, 0x27BF),
    (0x2934, 0x2935),
    (0x2B05, 0x2B07),
    (0x2B1B, 0x2B1C),
    (0x2B50, 0x2B50),
    (0x2B55, 0x2B55),
    (0x3030, 0x3030),
    (0x303D, 0x303D),
    (0x3297, 0x3297),
    (0x3299, 0x3299),
    (0x1F000, 0x1FAFF),
)

EMOJI_CONNECTORS = {0x200D, 0x20E3, 0xFE0E, 0xFE0F}
KEYCAP_BASE_CHARS = set("0123456789#*")


@dataclass(frozen=True)
class PeriodSpec:
    period_id: str
    period_label: str
    period_index: int
    start: pd.Timestamp
    end: pd.Timestamp

    @property
    def end_exclusive(self) -> pd.Timestamp:
        return self.end + pd.Timedelta(days=1)

    @property
    def total_days(self) -> int:
        return int((self.end - self.start).days + 1)


def build_period_specs() -> list[PeriodSpec]:
    return [
        PeriodSpec("p1", "2017-2019", 1, pd.Timestamp("2017-01-01"), pd.Timestamp("2019-12-31")),
        PeriodSpec("p2", "2020-2022", 2, pd.Timestamp("2020-01-01"), pd.Timestamp("2022-12-31")),
        PeriodSpec("p3", "2023-2026-03", 3, pd.Timestamp("2023-01-01"), pd.Timestamp("2026-03-31")),
    ]


def build_user_key(comments: pd.DataFrame) -> pd.Series:
    from_username = comments["from_username"].fillna("").astype("string").str.strip()
    from_username_norm = from_username.str.casefold()
    from_id = comments["from_id"].fillna("").astype("string").str.strip()
    from_id = from_id.mask(from_id == "")
    return from_id.fillna("username:" + from_username_norm)


def is_emojiish_char(char: str) -> bool:
    codepoint = ord(char)
    if codepoint in EMOJI_CONNECTORS or 0x1F3FB <= codepoint <= 0x1F3FF:
        return True
    for start, end in EMOJI_RANGES:
        if start <= codepoint <= end:
            return True
    return False


def is_regional_indicator(codepoint: int) -> bool:
    return 0x1F1E6 <= codepoint <= 0x1F1FF


def is_skin_tone_modifier(codepoint: int) -> bool:
    return 0x1F3FB <= codepoint <= 0x1F3FF


def is_emoji_base_char(char: str) -> bool:
    codepoint = ord(char)
    if codepoint in EMOJI_CONNECTORS or is_skin_tone_modifier(codepoint):
        return False
    if unicodedata.combining(char) > 0:
        return False
    return is_emojiish_char(char)


def is_combining_or_extender(char: str) -> bool:
    codepoint = ord(char)
    return (
        unicodedata.combining(char) > 0
        or codepoint in {0x20E3, 0xFE0E, 0xFE0F}
        or is_skin_tone_modifier(codepoint)
    )


def consume_keycap_sequence(value: str, start: int) -> int | None:
    if value[start] not in KEYCAP_BASE_CHARS:
        return None
    end = start + 1
    if end < len(value) and ord(value[end]) == 0xFE0F:
        end += 1
    if end < len(value) and ord(value[end]) == 0x20E3:
        return end + 1
    return None


def consume_emoji_cluster(value: str, start: int) -> tuple[str, int] | None:
    keycap_end = consume_keycap_sequence(value, start)
    if keycap_end is not None:
        return value[start:keycap_end], keycap_end
    if start >= len(value):
        return None
    char = value[start]
    codepoint = ord(char)
    if is_regional_indicator(codepoint):
        end = start + 1
        if end < len(value) and is_regional_indicator(ord(value[end])):
            end += 1
        return value[start:end], end
    if not is_emoji_base_char(char):
        return None
    end = start + 1
    while end < len(value) and is_combining_or_extender(value[end]):
        end += 1
    while end < len(value) and ord(value[end]) == 0x200D:
        end += 1
        if end >= len(value):
            break
        end += 1
        while end < len(value) and is_combining_or_extender(value[end]):
            end += 1
    return value[start:end], end


def count_words_with_emoji(text: object) -> int:
    if pd.isna(text):
        return 0
    value = str(text).strip()
    if not value:
        return 0
    count = 0
    last_emoji_cluster: str | None = None
    non_emoji_chunk: list[str] = []

    def flush_non_emoji() -> int:
        nonlocal non_emoji_chunk
        if not non_emoji_chunk:
            return 0
        piece = "".join(non_emoji_chunk)
        non_emoji_chunk = []
        return len(NON_EMOJI_TOKEN_RE.findall(piece))

    index = 0
    while index < len(value):
        char = value[index]
        if char.isspace():
            count += flush_non_emoji()
            last_emoji_cluster = None
            index += 1
            continue
        emoji_cluster = consume_emoji_cluster(value, index)
        if emoji_cluster is not None:
            count += flush_non_emoji()
            cluster_text, next_index = emoji_cluster
            if cluster_text != last_emoji_cluster:
                count += 1
                last_emoji_cluster = cluster_text
            index = next_index
            continue
        last_emoji_cluster = None
        non_emoji_chunk.append(char)
        index += 1
    count += flush_non_emoji()
    return count


## Build base inputs

We keep top-level non-creator comments in the full 2017 to March 2026 range, remove users with one comment, and keep the matching posts.

In [ ]:
def load_all_comments() -> pd.DataFrame:
    comments = pd.read_csv(COMMENTS_SOURCE, dtype=COMMENT_DTYPES, low_memory=False)
    comments["timestamp"] = pd.to_datetime(comments["timestamp"], errors="coerce")
    comments = comments.dropna(subset=["comment_id", "media_id", "timestamp"]).copy()
    comments["from_username"] = comments["from_username"].fillna("").astype("string").str.strip()
    comments["from_id"] = comments["from_id"].fillna("").astype("string").str.strip()
    comments["parent_id"] = comments["parent_id"].fillna("").astype("string").str.strip()
    comments["user_key"] = build_user_key(comments)
    return comments


def load_all_posts() -> pd.DataFrame:
    posts = pd.read_csv(POSTS_SOURCE, dtype=POST_DTYPES, low_memory=False)
    posts["timestamp"] = pd.to_datetime(posts["timestamp"], errors="coerce")
    posts = posts.dropna(subset=["media_id", "timestamp"]).copy()
    return posts


all_comments = load_all_comments()
all_posts = load_all_posts()

eligible_posts = all_posts.loc[
    (all_posts["timestamp"] >= ANALYSIS_START) & (all_posts["timestamp"] < ANALYSIS_END_EXCLUSIVE)
].copy()
eligible_media_ids = pd.Index(eligible_posts["media_id"].drop_duplicates())

creator_mask_all = (all_comments["is_creator"] == True) | (
    all_comments["from_username"].str.casefold() == CREATOR_USERNAME
)
history_comments = all_comments.loc[~creator_mask_all].copy()
history_comments["word_count"] = history_comments["text"].map(count_words_with_emoji).astype(int)
history_comments = history_comments.loc[history_comments["word_count"] > 0].copy()

base_comments = history_comments.loc[
    (history_comments["timestamp"] >= ANALYSIS_START) & (history_comments["timestamp"] < ANALYSIS_END_EXCLUSIVE)
].copy()
base_comments = base_comments.loc[base_comments["media_id"].isin(eligible_media_ids)].copy()

reply_mask = (base_comments["is_reply"] == True) | base_comments["parent_id"].ne("")
base_comments = base_comments.loc[~reply_mask].copy()

user_comment_counts = base_comments.groupby("user_key").size()
keep_user_keys = user_comment_counts[user_comment_counts > 1].index

base_comments = base_comments.loc[base_comments["user_key"].isin(keep_user_keys)].copy()
base_comments = base_comments.sort_values(["timestamp", "media_id", "comment_id"], kind="mergesort").reset_index(drop=True)

base_posts = eligible_posts.loc[eligible_posts["media_id"].isin(base_comments["media_id"].drop_duplicates())].copy()
base_posts = base_posts.sort_values(["timestamp", "media_id"], kind="mergesort").reset_index(drop=True)

comments_output = base_comments.drop(columns=["word_count"])
comments_output.to_csv(COMMENTS_OUTPUT, index=False, encoding="utf-8-sig")
base_posts.to_csv(POSTS_OUTPUT, index=False, encoding="utf-8-sig")

print(f"Saved {len(comments_output):,} rows to {COMMENTS_OUTPUT.name}")
print(f"Saved {len(base_posts):,} rows to {POSTS_OUTPUT.name}")
print(f"Users retained: {base_comments['user_key'].nunique():,}")
print(f"Posts retained: {base_posts['media_id'].nunique():,}")


## Compute period metrics

We calculate Recency, Coverage, Engagement, Delay, Tenure, and Gini for each user in each triennium.

In [ ]:
TARGET_COLUMNS = [
    "period_id",
    "period_label",
    "period_index",
    "period_start",
    "period_end",
    "period_total_days",
    "posts_in_period",
    "user_key",
    "user_id",
    "username",
    "comment_count",
    "last_comment_timestamp",
    "recency",
    "coverage",
    "engagement",
    "delay",
    "tenure",
    "gini",
]


def build_post_delay_reference(posts: pd.DataFrame, comments: pd.DataFrame) -> pd.DataFrame:
    first_comment_ts = (
        comments.groupby("media_id", sort=False)["timestamp"]
        .min()
        .rename("first_comment_timestamp")
        .reset_index()
    )
    post_reference = posts.merge(first_comment_ts, on="media_id", how="left")
    midnight_mask = (
        (post_reference["timestamp"].dt.hour == 0)
        & (post_reference["timestamp"].dt.minute == 0)
        & (post_reference["timestamp"].dt.second == 0)
    )
    can_backfill_mask = midnight_mask & post_reference["first_comment_timestamp"].notna()
    post_reference["delay_post_timestamp"] = post_reference["timestamp"]
    post_reference.loc[can_backfill_mask, "delay_post_timestamp"] = (
        post_reference.loc[can_backfill_mask, "first_comment_timestamp"] - MIDNIGHT_BACKOFF
    )
    return post_reference[["media_id", "delay_post_timestamp"]].copy()


def build_period_metrics(
    period: PeriodSpec,
    comments: pd.DataFrame,
    posts: pd.DataFrame,
    history: pd.DataFrame,
    first_comment_overall: pd.DataFrame,
    post_delay_reference: pd.DataFrame,
) -> pd.DataFrame:
    posts_mask = (posts["timestamp"] >= period.start) & (posts["timestamp"] < period.end_exclusive)
    period_posts = posts.loc[posts_mask].copy()
    posts_in_period = int(len(period_posts))
    if posts_in_period == 0:
        return pd.DataFrame(columns=TARGET_COLUMNS)

    period_post_ids = pd.Index(period_posts["media_id"].drop_duplicates())
    comments_mask = (comments["timestamp"] >= period.start) & (comments["timestamp"] < period.end_exclusive)
    period_comments = comments.loc[comments_mask & comments["media_id"].isin(period_post_ids)].copy()
    if period_comments.empty:
        return pd.DataFrame(columns=TARGET_COLUMNS)

    period_comments = period_comments.sort_values(["user_key", "timestamp", "comment_id"], kind="mergesort").reset_index(drop=True)
    posts_seen = np.searchsorted(
        period_posts["timestamp"].to_numpy(dtype="datetime64[ns]"),
        period_comments["timestamp"].to_numpy(dtype="datetime64[ns]"),
        side="right",
    )
    period_comments["posts_seen_by_comment_time"] = posts_seen

    period_comments = period_comments.merge(post_delay_reference, on="media_id", how="left")
    period_comments["delay_hours"] = (
        (period_comments["timestamp"] - period_comments["delay_post_timestamp"]) / pd.Timedelta(hours=1)
    )
    period_comments["delay_hours"] = period_comments["delay_hours"].clip(lower=0)

    per_post_counts = (
        period_comments.groupby(["user_key", "media_id"], sort=False)
        .size()
        .rename("comments_on_post")
        .reset_index()
    )
    per_post_counts["comment_share"] = per_post_counts["comments_on_post"] / per_post_counts.groupby("user_key")["comments_on_post"].transform("sum")
    gini = (
        per_post_counts.assign(comment_share_sq=lambda df: np.square(df["comment_share"]))
        .groupby("user_key", sort=False)["comment_share_sq"]
        .sum()
        .rename("hhi")
        .reset_index()
    )
    gini["gini"] = 1 - gini["hhi"]

    latest_history = (
        history.loc[history["timestamp"] < period.end_exclusive, ["user_key", "timestamp"]]
        .groupby("user_key", sort=False)["timestamp"]
        .max()
        .rename("latest_history_comment")
        .reset_index()
    )

    aggregated = (
        period_comments.groupby("user_key", sort=False)
        .agg(
            user_id=("from_id", "last"),
            username=("from_username", "last"),
            comment_count=("comment_id", "size"),
            distinct_posts_commented=("media_id", "nunique"),
            last_comment_timestamp=("timestamp", "max"),
            last_posts_seen=("posts_seen_by_comment_time", "last"),
            engagement=("word_count", "mean"),
            delay=("delay_hours", "mean"),
        )
        .reset_index()
    )

    aggregated = aggregated.merge(gini[["user_key", "gini"]], on="user_key", how="left")
    aggregated = aggregated.merge(first_comment_overall, on="user_key", how="left")
    aggregated = aggregated.merge(latest_history, on="user_key", how="left")

    aggregated["recency"] = (posts_in_period - aggregated["last_posts_seen"]) / posts_in_period
    aggregated["coverage"] = aggregated["distinct_posts_commented"] / posts_in_period
    aggregated["tenure"] = (
        (aggregated["latest_history_comment"] - aggregated["first_comment_timestamp"]) / pd.Timedelta(days=1)
    )

    aggregated["period_id"] = period.period_id
    aggregated["period_label"] = period.period_label
    aggregated["period_index"] = period.period_index
    aggregated["period_start"] = period.start.date().isoformat()
    aggregated["period_end"] = period.end.date().isoformat()
    aggregated["period_total_days"] = period.total_days
    aggregated["posts_in_period"] = posts_in_period

    metrics = aggregated[TARGET_COLUMNS].copy()
    for column in ["recency", "coverage", "engagement", "delay", "tenure", "gini"]:
        metrics[column] = metrics[column].round(6)
    return metrics


base_comments = pd.read_csv(COMMENTS_OUTPUT, low_memory=False)
base_comments["timestamp"] = pd.to_datetime(base_comments["timestamp"], errors="coerce")
base_comments["from_username"] = base_comments["from_username"].fillna("").astype("string").str.strip()
base_comments["from_id"] = base_comments["from_id"].fillna("").astype("string").str.strip()
base_comments["user_key"] = build_user_key(base_comments)
base_comments["word_count"] = base_comments["text"].map(count_words_with_emoji).astype(int)

base_posts = pd.read_csv(POSTS_OUTPUT, low_memory=False)
base_posts["timestamp"] = pd.to_datetime(base_posts["timestamp"], errors="coerce")

first_comment_overall = (
    base_comments.groupby("user_key", sort=False)["timestamp"]
    .min()
    .rename("first_comment_timestamp")
    .reset_index()
)

post_delay_reference = build_post_delay_reference(base_posts, base_comments)
periods = build_period_specs()

metrics_frames = [
    build_period_metrics(
        period=period,
        comments=base_comments,
        posts=base_posts,
        history=base_comments,
        first_comment_overall=first_comment_overall,
        post_delay_reference=post_delay_reference,
    )
    for period in periods
]

metrics = pd.concat(metrics_frames, ignore_index=True)
metrics = metrics.sort_values(["period_index", "user_key"], kind="mergesort").reset_index(drop=True)
metrics.to_csv(METRICS_OUTPUT, index=False, encoding="utf-8-sig")

print(f"Saved {len(metrics):,} rows to {METRICS_OUTPUT.name}")
print(f"Periods: {len(periods):,}")
print(f"First period: {periods[0].period_label}")
print(f"Last period: {periods[-1].period_label}")
metrics.head()


## Plot period distributions

We compare the three non-overlapping periods in one 2x3 grid to inspect the RCEDTG distributions before choosing thresholds.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PLOT_OUTPUT = OUTPUT_DIR / "ig_nonoverlapping_3y_periods_distributions.png"
PLOT_VARIABLES = [
    ("recency", "Recency"),
    ("coverage", "Coverage"),
    ("engagement", "Engagement"),
    ("delay", "Delay"),
    ("tenure", "Tenure"),
    ("gini", "Gini"),
]
PLOT_COLORS = ["#0B6E4F", "#C85C1A", "#3D6FB6"]
BOUNDED_RANGES = {
    "recency": (0.0, 1.0),
    "coverage": (0.0, 0.2),
    "engagement": (0.0, 20.0),
    "delay": (0.0, 200.0),
    "gini": (0.0, 1.0),
}
THRESHOLD_PERCENTILES = {
    "recency": (0.30, 0.70),
    "coverage": (0.60, 0.90),
}
TENURE_PERIOD_PERCENTILES = (0.30, 0.70)
FIXED_THRESHOLDS = {
    "gini": (0.2, 0.6),
}
TARGET_DELAY_THRESHOLDS = (6.0, 24.0)
BANDWIDTH_MULTIPLIERS = {
    "engagement": 1.15,
    "delay": 1.2,
}


def estimate_bandwidth(values: np.ndarray, multiplier: float = 1.0) -> float:
    values = values[np.isfinite(values)]
    if values.size <= 1:
        baseline = abs(values[0]) if values.size == 1 else 1.0
        return max(baseline * 0.05 * multiplier, 1e-3)
    std = np.std(values, ddof=1)
    iqr = np.subtract(*np.percentile(values, [75, 25]))
    sigma = min(std, iqr / 1.34) if iqr > 0 else std
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = max((values.max() - values.min()) / 6, 1e-3)
    return max(0.9 * sigma * values.size ** (-1 / 5) * multiplier, 1e-3)


def gaussian_kde_on_grid(values: np.ndarray, grid: np.ndarray, multiplier: float = 1.0) -> np.ndarray:
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.zeros_like(grid)
    bandwidth = estimate_bandwidth(values, multiplier=multiplier)
    scaled = (grid[:, None] - values[None, :]) / bandwidth
    kernel = np.exp(-0.5 * np.square(scaled)) / np.sqrt(2 * np.pi)
    return kernel.mean(axis=1) / bandwidth


def percentile_rank(values: np.ndarray, target: float) -> float:
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan")
    return float(np.mean(values <= target))


if "metrics" not in globals():
    metrics = pd.read_csv(METRICS_OUTPUT, low_memory=False)

thresholds_by_metric = {}
for column, percentiles in THRESHOLD_PERCENTILES.items():
    thresholds_by_metric[column] = tuple(float(metrics[column].quantile(percentile)) for percentile in percentiles)

for column, threshold_pair in FIXED_THRESHOLDS.items():
    thresholds_by_metric[column] = threshold_pair

thresholds_by_metric["engagement"] = (
    float(metrics["engagement"].quantile(0.60)),
    float(metrics["engagement"].quantile(0.90)),
)

delay_values = metrics["delay"].dropna().to_numpy(dtype=float)
delay_percentiles = tuple(percentile_rank(delay_values, target) for target in TARGET_DELAY_THRESHOLDS)
thresholds_by_metric["delay"] = tuple(
    float(metrics["delay"].quantile(percentile)) for percentile in delay_percentiles
)

tenure_thresholds_by_period = (
    metrics.groupby("period_label")["tenure"]
    .quantile(list(TENURE_PERIOD_PERCENTILES))
    .unstack()
    .rename(columns={TENURE_PERIOD_PERCENTILES[0]: "low", TENURE_PERIOD_PERCENTILES[1]: "high"})
    .reset_index()
)

plot_periods = (
    metrics[["period_label", "period_index"]]
    .drop_duplicates()
    .sort_values("period_index")
    .reset_index(drop=True)
)
period_color_lookup = dict(zip(plot_periods["period_label"], PLOT_COLORS))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.subplots_adjust(top=0.84, wspace=0.24, hspace=0.34)
axes = axes.ravel()

for axis, (column, label) in zip(axes, PLOT_VARIABLES):
    all_values = metrics[column].dropna().to_numpy(dtype=float)
    x_min = float(all_values.min())
    x_max = float(all_values.max())
    padding = 0.06 * (x_max - x_min) if x_max > x_min else max(abs(x_min) * 0.1, 1.0)
    lower = x_min - padding
    upper = x_max + padding
    if column in BOUNDED_RANGES:
        bound_low, bound_high = BOUNDED_RANGES[column]
        lower = max(lower, bound_low)
        upper = min(upper, bound_high)
    grid = np.linspace(lower, upper, 700)
    bandwidth_multiplier = BANDWIDTH_MULTIPLIERS.get(column, 1.0)

    for color, period in zip(PLOT_COLORS, plot_periods.itertuples(index=False)):
        period_values = metrics.loc[metrics["period_label"] == period.period_label, column].dropna().to_numpy(dtype=float)
        density = gaussian_kde_on_grid(period_values, grid, multiplier=bandwidth_multiplier)
        axis.plot(grid, density, linewidth=2.2, color=color, label=period.period_label)

    if column == "tenure":
        for row in tenure_thresholds_by_period.itertuples(index=False):
            color = period_color_lookup[row.period_label]
            axis.axvline(row.low, color=color, linestyle="--", linewidth=1.4, alpha=0.95)
            axis.axvline(row.high, color=color, linestyle="--", linewidth=1.4, alpha=0.95)
    else:
        low_threshold, high_threshold = thresholds_by_metric[column]
        axis.axvline(low_threshold, color="#333333", linestyle="--", linewidth=1.5)
        axis.axvline(high_threshold, color="#333333", linestyle="--", linewidth=1.5)

    axis.set_title(label)
    axis.set_xlabel(label)
    axis.set_ylabel("Density")
    axis.grid(alpha=0.2, linewidth=0.6)

handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle("Metric distributions across the three non-overlapping periods", fontsize=16, y=0.98)
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.93))
fig.savefig(PLOT_OUTPUT, dpi=220, bbox_inches="tight")
plt.close(fig)

for column, (low_threshold, high_threshold) in thresholds_by_metric.items():
    print(f"{column}: low={low_threshold:.6f}, high={high_threshold:.6f}")

print("tenure thresholds by period:")
print(tenure_thresholds_by_period.to_string(index=False))
print(f"Saved plot to {PLOT_OUTPUT.name}")


## Classify RCEDTG levels

We assign Low, Mid, and High labels to each metric using the selected thresholds and export one row per user-period.

In [ ]:
CLASSIFICATION_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_classification_nonoverlapping_3y.csv"
CLASSIFICATION_VARIABLES = ["recency", "coverage", "engagement", "delay", "tenure", "gini"]
PROFILE_LETTERS = {
    "recency": "R",
    "coverage": "C",
    "engagement": "E",
    "delay": "D",
    "tenure": "T",
    "gini": "G",
}


def assign_level(value: float, low_threshold: float, high_threshold: float) -> str:
    if pd.isna(value):
        return pd.NA
    if value <= low_threshold:
        return "Low"
    if value <= high_threshold:
        return "Mid"
    return "High"


if "metrics" not in globals():
    metrics = pd.read_csv(METRICS_OUTPUT, low_memory=False)

if "thresholds_by_metric" not in globals():
    thresholds_by_metric = {}
    for column, percentiles in THRESHOLD_PERCENTILES.items():
        thresholds_by_metric[column] = tuple(float(metrics[column].quantile(percentile)) for percentile in percentiles)
    for column, threshold_pair in FIXED_THRESHOLDS.items():
        thresholds_by_metric[column] = threshold_pair
    thresholds_by_metric["engagement"] = (
        float(metrics["engagement"].quantile(0.60)),
        float(metrics["engagement"].quantile(0.90)),
    )
    delay_values = metrics["delay"].dropna().to_numpy(dtype=float)
    delay_percentiles = tuple(percentile_rank(delay_values, target) for target in TARGET_DELAY_THRESHOLDS)
    thresholds_by_metric["delay"] = tuple(
        float(metrics["delay"].quantile(percentile)) for percentile in delay_percentiles
    )

if "tenure_thresholds_by_period" not in globals():
    tenure_thresholds_by_period = (
        metrics.groupby("period_label")["tenure"]
        .quantile(list(TENURE_PERIOD_PERCENTILES))
        .unstack()
        .rename(columns={TENURE_PERIOD_PERCENTILES[0]: "low", TENURE_PERIOD_PERCENTILES[1]: "high"})
        .reset_index()
    )

classification = metrics.copy()

for column in [value for value in CLASSIFICATION_VARIABLES if value != "tenure"]:
    low_threshold, high_threshold = thresholds_by_metric[column]
    classification[f"{column}_low_threshold"] = low_threshold
    classification[f"{column}_high_threshold"] = high_threshold
    classification[f"{column}_level"] = classification[column].map(
        lambda value, low=low_threshold, high=high_threshold: assign_level(value, low, high)
    )

classification = classification.merge(
    tenure_thresholds_by_period.rename(columns={"low": "tenure_low_threshold", "high": "tenure_high_threshold"}),
    on="period_label",
    how="left",
)
classification["tenure_level"] = classification.apply(
    lambda row: assign_level(row["tenure"], row["tenure_low_threshold"], row["tenure_high_threshold"]),
    axis=1,
)

classification["rcedtg_profile"] = classification.apply(
    lambda row: " | ".join(f"{PROFILE_LETTERS[column]}={row[f'{column}_level']}" for column in CLASSIFICATION_VARIABLES),
    axis=1,
)

classification.to_csv(CLASSIFICATION_OUTPUT, index=False, encoding="utf-8-sig")
print(f"Saved {len(classification):,} rows to {CLASSIFICATION_OUTPUT.name}")
classification.head()


## Run k-medoids on RCEDTG profiles

We cluster the classified user-period observations with weighted k-medoids on unique ordinal RCEDTG profiles, test k from 2 to 10, and select the best solution by weighted silhouette.

In [ ]:
from sklearn.decomposition import PCA

KMEDOIDS_K_SEARCH_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_k_search.csv"
KMEDOIDS_ASSIGNMENT_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_k4_assignment.csv"
KMEDOIDS_SUMMARY_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_k4_cluster_summary.csv"
KMEDOIDS_DESCRIPTION_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_k4_cluster_descriptions.csv"
KMEDOIDS_DIAGNOSTIC_PLOT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_diagnostics.png"
KMEDOIDS_CLUSTER_PLOT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_k4_clusters_2d.png"

CLUSTER_VARIABLES = ["recency", "coverage", "engagement", "delay", "tenure", "gini"]
LEVEL_COLUMNS = [f"{column}_level" for column in CLUSTER_VARIABLES]
LEVEL_MAP = {"Low": 0, "Mid": 1, "High": 2}
K_SELECTED = 4
SIGNATURE_METADATA = {
    "Low-Mid-Mid-Low-High-High": {
        "cluster_name": "Brand Advocates",
        "description": "The strongest group: very recent, broader in coverage, more expressive, fast to respond, highly tenured within period, and highly dispersed across posts.",
    },
    "Mid-Mid-Low-Low-Mid-High": {
        "cluster_name": "Active Supporters",
        "description": "Fairly recent users with broader post coverage, lower engagement, fast response, medium tenure, and strong dispersion across posts.",
    },
    "High-Low-Low-Low-Mid-Mid": {
        "cluster_name": "Quiet Regulars",
        "description": "Less recent but still established users with narrow coverage, low engagement, fast response, and medium concentration across posts.",
    },
    "Mid-Low-Low-Low-Low-Low": {
        "cluster_name": "Occasional Newcomers",
        "description": "The lightest and newest group: narrow in coverage and engagement, fast to respond, and low in tenure and dispersion.",
    },
}


def mode_value(series: pd.Series) -> object:
    modes = series.mode(dropna=True)
    return modes.iloc[0] if not modes.empty else pd.NA


def manhattan_distance_matrix(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X)
    return np.abs(X[:, None, :] - X[None, :, :]).sum(axis=2)


def initialize_medoids_kpp(D: np.ndarray, weights: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
    n = D.shape[0]
    weights = np.asarray(weights, dtype=float)
    first = rng.choice(n, p=weights / weights.sum())
    medoids = [first]
    while len(medoids) < k:
        dist_to_nearest = D[:, medoids].min(axis=1)
        probs = weights * np.square(dist_to_nearest)
        probs[medoids] = 0
        if probs.sum() == 0:
            remaining = [index for index in range(n) if index not in medoids]
            medoids.append(rng.choice(remaining))
        else:
            probs = probs / probs.sum()
            medoids.append(rng.choice(n, p=probs))
    return np.array(medoids, dtype=int)


def weighted_kmedoids(D: np.ndarray, weights: np.ndarray, k: int, n_init: int = 80, max_iter: int = 100, random_state: int = 42):
    rng = np.random.default_rng(random_state)
    n = D.shape[0]
    weights = np.asarray(weights, dtype=float)
    best_medoids = None
    best_labels = None
    best_cost = np.inf

    for _ in range(n_init):
        medoids = initialize_medoids_kpp(D, weights, k, rng)
        for _ in range(max_iter):
            distances_to_medoids = D[:, medoids]
            labels = distances_to_medoids.argmin(axis=1)
            new_medoids = medoids.copy()
            for cluster_id in range(k):
                members = np.where(labels == cluster_id)[0]
                if len(members) == 0:
                    nearest_dist = D[:, medoids].min(axis=1)
                    candidate_scores = weights * nearest_dist
                    candidate_scores[medoids] = -np.inf
                    new_medoids[cluster_id] = int(np.argmax(candidate_scores))
                    continue
                subD = D[np.ix_(members, members)]
                subW = weights[members]
                candidate_costs = subW @ subD
                new_medoids[cluster_id] = members[np.argmin(candidate_costs)]
            new_medoids = np.array(new_medoids, dtype=int)
            if np.array_equal(np.sort(new_medoids), np.sort(medoids)):
                medoids = new_medoids
                break
            medoids = new_medoids
        labels = D[:, medoids].argmin(axis=1)
        cost = np.sum(weights * D[np.arange(n), medoids[labels]])
        if cost < best_cost:
            best_cost = cost
            best_medoids = medoids.copy()
            best_labels = labels.copy()
    return best_medoids, best_labels, best_cost


def weighted_silhouette_score(D: np.ndarray, labels: np.ndarray, weights: np.ndarray) -> float:
    labels = np.asarray(labels)
    weights = np.asarray(weights, dtype=float)
    unique_labels = np.unique(labels)
    s = np.zeros(len(labels), dtype=float)
    for index in range(len(labels)):
        own = labels[index]
        same_mask = labels == own
        same_weight = weights[same_mask].sum()
        if same_weight <= 1:
            a = 0.0
        else:
            a = np.sum(weights[same_mask] * D[index, same_mask]) / (same_weight - 1)
        b_values = []
        for other in unique_labels:
            if other == own:
                continue
            other_mask = labels == other
            other_weight = weights[other_mask].sum()
            if other_weight > 0:
                b_values.append(np.sum(weights[other_mask] * D[index, other_mask]) / other_weight)
        if not b_values:
            s[index] = 0.0
        else:
            b = min(b_values)
            denom = max(a, b)
            s[index] = 0.0 if denom == 0 else (b - a) / denom
    return float(np.average(s, weights=weights))


def ordering_score(signature: tuple[str, ...]) -> int:
    recency, coverage, engagement, delay, tenure, gini = signature
    score_map = {"Low": 0, "Mid": 1, "High": 2}
    return (
        -score_map[recency]
        + score_map[coverage]
        + score_map[engagement]
        - score_map[delay]
        + score_map[tenure]
        + score_map[gini]
    )


if "classification" not in globals():
    classification = pd.read_csv(CLASSIFICATION_OUTPUT, low_memory=False)

profile_counts = (
    classification[LEVEL_COLUMNS]
    .value_counts()
    .reset_index(name="weight")
    .sort_values(LEVEL_COLUMNS)
    .reset_index(drop=True)
)

X_unique = profile_counts[LEVEL_COLUMNS].replace(LEVEL_MAP).astype(int).to_numpy()
weights = profile_counts["weight"].to_numpy(dtype=float)
D_unique = manhattan_distance_matrix(X_unique)

evaluation_rows = []
fitted_models = {}
for k in range(2, 11):
    medoids, labels_unique, cost = weighted_kmedoids(
        D=D_unique,
        weights=weights,
        k=k,
        n_init=100,
        max_iter=100,
        random_state=42,
    )
    silhouette = weighted_silhouette_score(D_unique, labels_unique, weights)
    cluster_sizes = pd.DataFrame({"cluster": labels_unique, "weight": weights}).groupby("cluster")["weight"].sum()
    fitted_models[k] = (medoids, labels_unique)
    evaluation_rows.append({
        "k": k,
        "weighted_silhouette_score": float(silhouette),
        "total_weighted_distance": float(cost),
        "min_cluster_size": int(cluster_sizes.min()),
        "max_cluster_size": int(cluster_sizes.max()),
        "smallest_cluster_share": float(cluster_sizes.min() / weights.sum()),
        "largest_cluster_share": float(cluster_sizes.max() / weights.sum()),
    })

evaluation_df = pd.DataFrame(evaluation_rows).sort_values("k").reset_index(drop=True)
evaluation_df.to_csv(KMEDOIDS_K_SEARCH_OUTPUT, index=False, encoding="utf-8-sig")

best_k = int(evaluation_df.loc[evaluation_df["weighted_silhouette_score"].idxmax(), "k"])
selected_medoids, selected_labels_unique = fitted_models[K_SELECTED]

profile_counts["cluster_raw"] = [f"R{label + 1}" for label in selected_labels_unique]
raw_modes = profile_counts.groupby("cluster_raw", sort=True)[LEVEL_COLUMNS].agg(mode_value).reset_index()
raw_modes["signature"] = raw_modes[LEVEL_COLUMNS].apply(tuple, axis=1)
raw_modes["ordering_score"] = raw_modes["signature"].map(ordering_score)
raw_modes = raw_modes.sort_values(["ordering_score", "cluster_raw"], ascending=[False, True]).reset_index(drop=True)

cluster_lookup_rows = []
for cluster_index, row in enumerate(raw_modes.itertuples(index=False), start=1):
    cluster_lookup_rows.append({
        "cluster_raw": row.cluster_raw,
        "cluster": f"C{cluster_index}",
    })

cluster_lookup = pd.DataFrame(cluster_lookup_rows)

clustered = classification.merge(profile_counts[LEVEL_COLUMNS + ["cluster_raw"]], on=LEVEL_COLUMNS, how="left")
clustered = clustered.merge(cluster_lookup, on="cluster_raw", how="left")
clustered = clustered.drop(columns=["cluster_raw"])

summary_modes = clustered.groupby("cluster", sort=True)[LEVEL_COLUMNS].agg(mode_value)
summary_sizes = clustered.groupby("cluster", sort=True).size().rename("n_obs")
summary_sizes = summary_sizes.to_frame()
summary_sizes["share_obs"] = summary_sizes["n_obs"] / len(clustered)
summary_medians = clustered.groupby("cluster", sort=True)[CLUSTER_VARIABLES].median().add_prefix("median_")
cluster_summary = summary_sizes.join(summary_modes).join(summary_medians).reset_index()
cluster_summary["rcedtg_signature"] = cluster_summary[[
    "recency_level", "coverage_level", "engagement_level", "delay_level", "tenure_level", "gini_level"
]].agg("-".join, axis=1)
cluster_summary["cluster_name"] = cluster_summary["rcedtg_signature"].map(lambda value: SIGNATURE_METADATA.get(value, {}).get("cluster_name", value))
cluster_summary["description"] = cluster_summary["rcedtg_signature"].map(lambda value: SIGNATURE_METADATA.get(value, {}).get("description", "Fallback description for an unmapped profile."))
cluster_summary = cluster_summary.sort_values("cluster").reset_index(drop=True)
name_lookup = cluster_summary[["cluster", "cluster_name", "description", "rcedtg_signature"]].copy()
clustered = clustered.merge(name_lookup, on="cluster", how="left")
clustered.to_csv(KMEDOIDS_ASSIGNMENT_OUTPUT, index=False, encoding="utf-8-sig")
cluster_summary.to_csv(KMEDOIDS_SUMMARY_OUTPUT, index=False, encoding="utf-8-sig")
name_lookup.to_csv(KMEDOIDS_DESCRIPTION_OUTPUT, index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(evaluation_df["k"], evaluation_df["total_weighted_distance"], marker="o", linewidth=2.2, color="#0B6E4F")
axes[0].axvline(K_SELECTED, linestyle="--", linewidth=1.3, color="#111111")
axes[0].set_title("Weighted medoid cost")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Total weighted distance")
axes[0].grid(alpha=0.2, linewidth=0.6)

axes[1].plot(evaluation_df["k"], evaluation_df["weighted_silhouette_score"], marker="o", linewidth=2.2, color="#C85C1A")
axes[1].axvline(best_k, linestyle="--", linewidth=1.3, color="#666666")
axes[1].axvline(K_SELECTED, linestyle="--", linewidth=1.3, color="#111111")
axes[1].set_title("Weighted silhouette score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Score")
axes[1].grid(alpha=0.2, linewidth=0.6)

fig.suptitle("Weighted k-medoids diagnostics for RCEDTG profiles", fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(KMEDOIDS_DIAGNOSTIC_PLOT, dpi=220, bbox_inches="tight")
plt.close(fig)

X_full = classification[LEVEL_COLUMNS].replace(LEVEL_MAP).astype(int)
pca = PCA(n_components=2, random_state=42)
projection = pca.fit_transform(X_full)
medoid_profiles = profile_counts.iloc[selected_medoids][LEVEL_COLUMNS].replace(LEVEL_MAP).astype(int)
centers_2d = pca.transform(medoid_profiles)
plot_df = pd.DataFrame({
    "pc1": projection[:, 0],
    "pc2": projection[:, 1],
    "cluster": clustered["cluster"],
})

cluster_labels_sorted = sorted(plot_df["cluster"].unique(), key=lambda value: int(value[1:]))
cluster_palette = ["#0B6E4F", "#C85C1A", "#3D6FB6", "#8E6CBE", "#D84A6A", "#6D7F4B", "#9C6B30", "#4C8C8A", "#B46CC2"]
cluster_color_map = {cluster: cluster_palette[index] for index, cluster in enumerate(cluster_labels_sorted)}
medoid_cluster_order = (
    cluster_lookup.assign(raw_order=cluster_lookup["cluster_raw"].str[1:].astype(int))
    .sort_values("raw_order")
    ["cluster"]
    .tolist()
)
centers_df = pd.DataFrame({
    "cluster": medoid_cluster_order,
    "pc1": centers_2d[:, 0],
    "pc2": centers_2d[:, 1],
}).sort_values("cluster")

fig, axis = plt.subplots(figsize=(8.8, 6.4))
for cluster in cluster_labels_sorted:
    subset = plot_df.loc[plot_df["cluster"] == cluster]
    axis.scatter(
        subset["pc1"],
        subset["pc2"],
        s=12,
        alpha=0.28,
        color=cluster_color_map[cluster],
        label=cluster,
        edgecolors="none",
    )

for row in centers_df.itertuples(index=False):
    axis.scatter(
        row.pc1,
        row.pc2,
        s=220,
        marker="X",
        color="#111111",
        edgecolors="white",
        linewidths=1.0,
        zorder=5,
    )
    axis.text(row.pc1, row.pc2, f" {row.cluster}", va="center", ha="left", fontsize=10, color="#111111")

axis.set_title(f"Weighted k-medoids clusters in 2D PCA space (k={K_SELECTED})")
axis.set_xlabel("PC1")
axis.set_ylabel("PC2")
axis.grid(alpha=0.18, linewidth=0.6)
axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=min(K_SELECTED, 5), frameon=False)
fig.tight_layout()
fig.savefig(KMEDOIDS_CLUSTER_PLOT, dpi=220, bbox_inches="tight")
plt.close(fig)

selected_silhouette = float(evaluation_df.loc[evaluation_df['k'] == K_SELECTED, 'weighted_silhouette_score'].iloc[0])
best_silhouette = float(evaluation_df.loc[evaluation_df['k'] == best_k, 'weighted_silhouette_score'].iloc[0])
print(f"Best k by weighted silhouette: {best_k}")
print(f"Best weighted silhouette: {best_silhouette:.6f}")
print(f"Selected k: {K_SELECTED}")
print(f"Selected weighted silhouette: {selected_silhouette:.6f}")
print(f"Saved search table to {KMEDOIDS_K_SEARCH_OUTPUT.name}")
print(f"Saved assignments to {KMEDOIDS_ASSIGNMENT_OUTPUT.name}")
print(f"Saved cluster summary to {KMEDOIDS_SUMMARY_OUTPUT.name}")
evaluation_df


## Plot selected cluster composition by period

We compare the selected weighted k-medoids solution across the three periods with absolute counts and within-period cluster shares.

In [ ]:
COUNTS_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_k4_counts_by_period.csv"
SHARES_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_k4_shares_by_period_pct.csv"
BAR_PLOT_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_k4_absolute_counts_bar.png"
PIE_PLOT_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_nonoverlapping_3y_kmedoids_k4_shares_pies.png"
PLOT_CLUSTER_COLORS = {
    "C1": "#2A9D8F",
    "C2": "#E76F51",
    "C3": "#264653",
    "C4": "#E9C46A",
}


if "clustered" not in globals():
    clustered = pd.read_csv(KMEDOIDS_ASSIGNMENT_OUTPUT, low_memory=False)

period_order = (
    clustered[["period_label", "period_index"]]
    .drop_duplicates()
    .sort_values("period_index")
    ["period_label"]
    .tolist()
)

if "cluster_summary" in globals():
    plot_cluster_order = cluster_summary.sort_values("cluster")["cluster"].tolist()
    plot_cluster_labels = {
        row.cluster: f"{row.cluster} {row.cluster_name}"
        for row in cluster_summary[["cluster", "cluster_name"]].itertuples(index=False)
    }
else:
    plot_cluster_order = sorted(clustered["cluster"].dropna().unique(), key=lambda value: int(value[1:]))
    plot_cluster_labels = {cluster: cluster for cluster in plot_cluster_order}

counts_by_period = (
    clustered.groupby(["period_label", "cluster"], sort=False)
    .size()
    .rename("user_period_count")
    .reset_index()
)
counts_pivot = (
    counts_by_period.pivot(index="period_label", columns="cluster", values="user_period_count")
    .reindex(index=period_order, columns=plot_cluster_order)
    .fillna(0)
    .astype(int)
)
counts_pivot.to_csv(COUNTS_OUTPUT, encoding="utf-8-sig")

shares_pivot = counts_pivot.div(counts_pivot.sum(axis=1), axis=0).multiply(100)
shares_pivot.to_csv(SHARES_OUTPUT, encoding="utf-8-sig")

fig, axis = plt.subplots(figsize=(16, 8.5))
x_positions = np.arange(len(period_order))
bar_width = 0.16
offsets = (np.arange(len(plot_cluster_order)) - (len(plot_cluster_order) - 1) / 2) * bar_width

for index, cluster in enumerate(plot_cluster_order):
    values = counts_pivot[cluster].to_numpy(dtype=float)
    bars = axis.bar(
        x_positions + offsets[index],
        values,
        width=bar_width,
        color=PLOT_CLUSTER_COLORS[cluster],
        edgecolor="white",
        linewidth=0.8,
        label=plot_cluster_labels[cluster],
    )
    axis.bar_label(bars, labels=[f"{int(value)}" for value in values], padding=3, fontsize=9, rotation=90)

axis.set_xticks(x_positions)
axis.set_xticklabels(period_order)
axis.set_ylabel("User-period count")
axis.set_title("Instagram RCEDTG k-medoids Cluster Counts by Period", fontsize=22)
axis.grid(axis="y", alpha=0.22, linewidth=0.9)
axis.set_axisbelow(True)
axis.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
fig.tight_layout()
fig.savefig(BAR_PLOT_OUTPUT, dpi=220, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(1, len(period_order), figsize=(20, 8.2))
for axis, period_label in zip(axes, period_order):
    shares = shares_pivot.loc[period_label, plot_cluster_order]
    wedges, texts, autotexts = axis.pie(
        shares,
        labels=None,
        colors=[PLOT_CLUSTER_COLORS[cluster] for cluster in plot_cluster_order],
        startangle=90,
        counterclock=False,
        autopct=lambda pct: f"{pct:.1f}%",
        wedgeprops={"linewidth": 1.1, "edgecolor": "white"},
        textprops={"fontsize": 10},
    )
    axis.set_title(period_label, fontsize=20, fontweight="bold", pad=18)
    for autotext in autotexts:
        autotext.set_color("#111111")
        autotext.set_fontsize(10)

fig.suptitle("Instagram RCEDTG k-medoids Cluster Shares Within Each Period", fontsize=26, y=0.97)
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', label=plot_cluster_labels[cluster], markerfacecolor=PLOT_CLUSTER_COLORS[cluster], markersize=14)
    for cluster in plot_cluster_order
]
fig.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 0.02), ncol=2, frameon=False, fontsize=12)
fig.tight_layout(rect=(0, 0.08, 1, 0.92))
fig.savefig(PIE_PLOT_OUTPUT, dpi=220, bbox_inches="tight")
plt.close(fig)

print(f"Saved counts table to {COUNTS_OUTPUT.name}")
print(f"Saved shares table to {SHARES_OUTPUT.name}")
print(f"Saved bar plot to {BAR_PLOT_OUTPUT.name}")
print(f"Saved pie plot to {PIE_PLOT_OUTPUT.name}")
counts_pivot


## Plot cluster radar charts by period

We compare the three period profiles within each selected cluster using globally scaled metrics and an inverted recency axis for interpretability.

In [ ]:
RADAR_DIR = OUTPUT_DIR / "cluster_radar_plots_nonoverlapping_3y"
RADAR_DIR.mkdir(parents=True, exist_ok=True)

RADAR_RAW_OUTPUT = RADAR_DIR / "ig_nonoverlapping_3y_kmedoids_k4_cluster_period_profiles_raw.csv"
RADAR_SCALED_OUTPUT = RADAR_DIR / "ig_nonoverlapping_3y_kmedoids_k4_cluster_period_profiles_scaled.csv"
RADAR_PLOT_OUTPUT = RADAR_DIR / "ig_nonoverlapping_3y_kmedoids_k4_cluster_radar_by_period.png"

RADAR_VARIABLES = ["recency", "coverage", "engagement", "delay", "tenure", "gini"]
RADAR_PERIOD_COLORS = {
    "2017-2019": "#1f77b4",
    "2020-2022": "#ff7f0e",
    "2023-2026-03": "#2ca02c",
}
RADAR_LABELS = {
    "recency_inverted_scaled": "Recency (inv)",
    "coverage_scaled": "Coverage",
    "engagement_scaled": "Engagement",
    "delay_scaled": "Delay",
    "tenure_scaled": "Tenure",
    "gini_scaled": "Gini",
}
RADAR_PLOT_COLUMNS = list(RADAR_LABELS.keys())

if "clustered" not in globals():
    clustered = pd.read_csv(KMEDOIDS_ASSIGNMENT_OUTPUT, low_memory=False)

if "cluster_summary" in globals():
    radar_cluster_order = cluster_summary.sort_values("cluster")["cluster"].tolist()
    radar_cluster_name_lookup = dict(cluster_summary[["cluster", "cluster_name"]].itertuples(index=False))
else:
    radar_cluster_order = sorted(clustered["cluster"].dropna().unique(), key=lambda value: int(value[1:]))
    radar_cluster_name_lookup = {cluster: cluster for cluster in radar_cluster_order}

radar_period_order = (
    clustered[["period_label", "period_index"]]
    .drop_duplicates()
    .sort_values("period_index")
    ["period_label"]
    .tolist()
)

radar_raw = (
    clustered.groupby(["cluster", "cluster_name", "period_label", "period_index"], sort=False)[RADAR_VARIABLES]
    .median()
    .reset_index()
)
radar_raw["cluster_order"] = radar_raw["cluster"].str[1:].astype(int)
radar_raw["cluster_label"] = radar_raw["cluster"] + " " + radar_raw["cluster_name"]
radar_raw = radar_raw.sort_values(["cluster_order", "period_index"]).reset_index(drop=True)
radar_raw.to_csv(RADAR_RAW_OUTPUT, index=False, encoding="utf-8-sig")

radar_scaled = radar_raw.copy()
for column in RADAR_VARIABLES:
    values = radar_scaled[column].to_numpy(dtype=float)
    min_value = float(np.min(values))
    max_value = float(np.max(values))
    if np.isclose(min_value, max_value):
        scaled_values = np.ones_like(values, dtype=float) * 0.5
    else:
        scaled_values = (values - min_value) / (max_value - min_value)
    radar_scaled[f"{column}_scaled"] = scaled_values

radar_scaled["recency_inverted_scaled"] = 1.0 - radar_scaled["recency_scaled"]

export_columns = [
    "cluster",
    "cluster_order",
    "cluster_name",
    "cluster_label",
    "period_label",
    "period_index",
    *RADAR_VARIABLES,
    *(f"{column}_scaled" for column in RADAR_VARIABLES),
    "recency_inverted_scaled",
]
radar_scaled[export_columns].to_csv(RADAR_SCALED_OUTPUT, index=False, encoding="utf-8-sig")

angles = np.linspace(0, 2 * np.pi, len(RADAR_PLOT_COLUMNS), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(2, 2, figsize=(15, 13), subplot_kw={"polar": True})
axes = axes.ravel()

for axis, cluster in zip(axes, radar_cluster_order):
    cluster_subset = radar_scaled.loc[radar_scaled["cluster"] == cluster].sort_values("period_index")
    for period_label in radar_period_order:
        period_row = cluster_subset.loc[cluster_subset["period_label"] == period_label]
        if period_row.empty:
            continue
        values = period_row.iloc[0][RADAR_PLOT_COLUMNS].to_numpy(dtype=float).tolist()
        values += values[:1]
        color = RADAR_PERIOD_COLORS[period_label]
        axis.plot(angles, values, color=color, linewidth=2.2, label=period_label)
        axis.fill(angles, values, color=color, alpha=0.10)

    axis.set_theta_offset(np.pi / 2)
    axis.set_theta_direction(-1)
    axis.set_xticks(angles[:-1])
    axis.set_xticklabels([RADAR_LABELS[column] for column in RADAR_PLOT_COLUMNS], fontsize=11)
    axis.set_ylim(0, 1)
    axis.set_yticks([0.25, 0.50, 0.75, 1.00])
    axis.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=10, color="#666666")
    axis.grid(alpha=0.28, linewidth=0.9)
    axis.set_title(f"{cluster}\n{radar_cluster_name_lookup[cluster]}", y=1.12, fontsize=16, fontweight="bold")

for axis in axes[len(radar_cluster_order):]:
    axis.remove()

legend_handles = [
    plt.Line2D([0], [0], color=RADAR_PERIOD_COLORS[period_label], linewidth=2.4, label=period_label)
    for period_label in radar_period_order
]
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.98), ncol=3, frameon=False, fontsize=13)
fig.suptitle("RCEDTG cluster median profiles by period", fontsize=24, y=1.03)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(RADAR_PLOT_OUTPUT, dpi=220, bbox_inches="tight")
plt.close(fig)

print(f"Saved raw radar data to {RADAR_RAW_OUTPUT.name}")
print(f"Saved scaled radar data to {RADAR_SCALED_OUTPUT.name}")
print(f"Saved radar plot to {RADAR_PLOT_OUTPUT.name}")
radar_scaled[export_columns].head()
